# L3.5 · ACT en uso: ejecutar los bloques


### El mapa de toda la serie

ACT es un **VAE condicional**:

- **Encoder del VAE** (sólo al entrenar): lee el estado y el bloque de acciones
  verdadero y lo resume en una variable `z`, el "estilo" de esa demostración.
- **Decoder = la policy** (al entrenar y al usar): un transformer
  encoder-decoder al estilo DETR. Lee la cámara, el estado y la `z`, y escribe
  el bloque entero de acciones de una pasada.
- **Al usarlo:** el encoder del VAE se tira y `z = 0`.

| Notebook | Qué abre |
|---|---|
| L3.1 | Los datos: de la grabación al ejemplo de entrenamiento |
| L3.2 | El encoder del VAE y la `z` |
| L3.3 | El decoder: la policy por dentro |
| L3.4 | El entrenamiento: un paso desarmado, y el modelo aprendiendo |
| L3.5 | El modelo en uso: ejecutar los bloques |

Todo usa el código real de LeRobot 0.4.1 y **nuestro modelo entrenado**
(`act_cheap_v1`), desarmado con `toolkit/policy/act_internals.py`. Sus tests
verifican que las piezas encadenadas dan exactamente lo mismo que el modelo.

In [ ]:
import torch  # antes que lerobot: evita un import circular entre torchvision y torch._dynamo
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lerobot.datasets.lerobot_dataset import LeRobotDataset

from toolkit.paths import CHECKPOINTS_DIR, DATASETS_DIR, REPO_ROOT
from toolkit.policy import act_internals as act
from toolkit.policy.load import load_policy

plt.rcParams["figure.dpi"] = 110
torch.manual_seed(0)
ROOT = DATASETS_DIR / "local" / "pick_mameluco_act_v1"
CAMERA = "observation.images.cam_right_high"
loaded = load_policy(CHECKPOINTS_DIR / "act_cheap_v1" / "pretrained_model", device="cpu")
policy, model, config = loaded.policy, loaded.policy.model, loaded.policy.config
_ = policy.eval()

## 1. Qué queda al usar el modelo

- El **encoder del VAE se tira**: no hay acciones verdaderas para darle.
- `z = 0`: el centro de la campana estándar, el "estilo promedio" de la
  distribución (que no es lo mismo que el promedio de las acciones).
- Una **predicción** es una sola pasada por la red, y devuelve el bloque
  entero: 48 × 26.

In [ ]:
import time

plain = LeRobotDataset("local/pick_mameluco_act_v1", root=ROOT, video_backend="pyav")
frame = plain[3000]
obs = loaded.preprocessor({k: frame[k] for k in ["observation.state", CAMERA]})
with torch.no_grad():
    policy.predict_action_chunk(obs)
    start = time.time()
    for _ in range(10):
        chunk = policy.predict_action_chunk(obs)
print(f"una predicción: {(time.time() - start) * 100:.0f} ms en la CPU de la Mac → bloque {tuple(chunk.shape)}")
print("el robot tarda", config.chunk_size / 30, "s en ejecutar ese bloque a 30 por segundo")

En el Jetson del robot medimos 312 ms por predicción: alcanza, porque el
bloque dura 1,6 s.

## 2. Tres formas de ejecutar los bloques

1. **Ejecutar las 48 y recién ahí volver a preguntar** (`n_action_steps = 48`,
   lo que hace nuestro modelo). Una consulta cada 1,6 s.
2. **Ejecutar sólo las primeras n** (`n_action_steps = n`). Reacciona más
   rápido; cuesta más consultas.
3. ***Temporal ensembling*** (el paper): preguntar en **cada** paso, y para la
   acción de ahora promediar todas las predicciones que hay para este instante,
   con más peso a las más viejas (pesos exp(−0,01·i)). Una consulta cada 33 ms.

Las comparamos sobre un episodio no visto, dándole siempre la observación
real (tarda unos 20 s).

In [ ]:
from lerobot.policies.act.modeling_act import ACTTemporalEnsembler

split = json.loads((REPO_ROOT / "learning" / "act" / "split.json").read_text())
episode = split["heldout_episode_indices"][2]
start, end = int(plain.meta.episodes[episode]["dataset_from_index"]), int(plain.meta.episodes[episode]["dataset_to_index"])
chunks = []
with torch.no_grad():
    for i in range(start, end):
        f = plain[i]
        chunks.append(policy.predict_action_chunk(loaded.preprocessor({k: f[k] for k in ["observation.state", CAMERA]})))
truth = np.stack(plain.hf_dataset.with_format("numpy")[start:end]["action"])
T = len(chunks)

def execute_every(n):
    out = []
    for t in range(0, T, n):
        out.extend(loaded.postprocessor(chunks[t])[0, :n].numpy())
    return np.array(out[:T])

ensembler = ACTTemporalEnsembler(temporal_ensemble_coeff=0.01, chunk_size=config.chunk_size)
ensembled = np.array([loaded.postprocessor(ensembler.update(c))[0].numpy() for c in chunks])
strategies = {"bloques de 48": execute_every(48), "bloques de 8": execute_every(8), "temporal ensembling": ensembled}

names = plain.meta.features["action"]["names"][0]
j = names.index("kRightElbow")
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.plot(truth[:, j], color="0.6", lw=4, label="operador")
for label, seq in strategies.items():
    ax.plot(seq[:, j], lw=1.2, label=label)
ax.set_title(f"episodio no visto {episode} · kRightElbow"); ax.set_xlabel("cuadro"); ax.legend()
plt.tight_layout()
for label, seq in strategies.items():
    print(f"{label:20s} error medio {np.abs(seq - truth).mean():.4f} rad · salto máximo entre cuadros {np.abs(np.diff(seq, axis=0)).max():.3f} rad")

- **Bloques de 48**: escalones donde un bloque termina y el siguiente arranca
  en otro lado. En este episodio, el primer bloque arranca el movimiento antes
  de tiempo (el operador se quedó quieto al principio) y hay un salto grande.
  En el robot, eso es un tirón.
- **Bloques de 8**: sigue mejor, con un serrucho de saltos chicos.
- **Temporal ensembling**: lo más suave por lejos.

Es *teacher forcing*: siempre la observación real. En el robot, la suavidad
cambia además lo que la cámara ve después; la decisión final es en lazo
cerrado.

## 3. Las perillas

In [ ]:
from lerobot.policies.act.configuration_act import ACTConfig

defaults = ACTConfig()
knobs = {
    "chunk_size": "acciones por predicción",
    "n_action_steps": "cuántas se ejecutan antes de volver a preguntar",
    "temporal_ensemble_coeff": "None = apagado; 0.01 = el del paper",
    "use_vae": "usar o no el encoder del VAE",
    "kl_weight": "cuánto cobra el KL",
    "latent_dim": "tamaño de z",
    "dim_model": "tamaño de los tokens",
    "n_heads": "cabezas de atención",
    "dim_feedforward": "capa interna de cada bloque",
    "n_encoder_layers": "capas del encoder",
    "n_decoder_layers": "capas del decoder",
    "dropout": "regularización",
    "optimizer_lr": "tasa de aprendizaje del transformer",
    "optimizer_lr_backbone": "tasa de aprendizaje de la ResNet",
}
pd.DataFrame({"nuestro": {k: getattr(config, k) for k in knobs},
              "LeRobot por defecto": {k: getattr(defaults, k) for k in knobs},
              "qué hace": knobs})

## Referencias

- Zhao et al. 2023, el paper de ACT y ALOHA:
  [arXiv 2304.13705](https://arxiv.org/abs/2304.13705). Arquitectura en la
  sección IV y el apéndice C; ablaciones en la sección VI.
- Código: `lerobot/policies/act/modeling_act.py` y `configuration_act.py`
  (dentro de `.venv`).
- DETR, de donde salen los casilleros del decoder:
  [arXiv 2005.12872](https://arxiv.org/abs/2005.12872).
- Desarmado paso a paso: `toolkit/policy/act_internals.py`.